# 2f, Koszul Jacobi $\Leftrightarrow [\pi,\pi]_{SN}=0$

## Problem

On 1-forms the bracket associated with a Poisson bivector $\pi$ is

$$
[\alpha,\beta]_K \;=\; \mathcal L_{\pi^\sharp\alpha}\beta \;-\; \mathcal L_{\pi^\sharp\beta}\alpha \;-\; d\bigl\langle\pi^\sharp\alpha,\beta\bigr\rangle,
$$

and the form-side Jacobi identity is

$$
\sum_{\text{cyc}}\bigl[\alpha,[\beta,\gamma]_K\bigr]_K
\;=\; 0
\qquad\Longleftrightarrow\qquad [\pi,\pi]_{SN}\;=\;0.
$$

## SN-bracket definition (the formula we cite)

$$
\tfrac12\,[\pi,\pi]_{SN}(\alpha,\beta,\gamma)
\;:=\;\sum_{\text{cyc}}\,\pi\bigl(\alpha,\,d\,\pi(\beta,\gamma)\bigr).
$$

The cyclic Koszul Jacobi sum on $(\alpha,\beta,\gamma)$ is precisely this combinator, the universal $[\pi,\pi]_{SN}$ handle in 1-form clothing.

## Strategy, Derived Bracket Theorem (no residue)

The Koszul bracket on 1-forms is the derived bracket of $\pi$ with respect to the SN bracket, anchored along $\pi^\sharp$:

$$
[\alpha,\beta]_K \;=\; \bigl[[\alpha,\pi]_{SN},\beta\bigr]_{SN}
\quad\text{acting via}\;\;\pi^\sharp.
$$

The **Derived Bracket Theorem** then gives the form-side reduction in a single step: the cyclic Jacobi sum on $(\alpha,\beta,\gamma)$ rewrites to the bare SN handle $[\pi,\pi]_{SN}$, the same handle as the function side (2g), with no expansion through Lie-derivative or pairing residues.

We obtain the chain via `PoissonBracket.prove_koszul_jacobi_reduction(α, β, γ)`.

In [1]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'jacopy' / '__init__.py').is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.brackets.base import BracketApply
from jacopy.brackets.schouten import sn as default_sn
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.poisson import PoissonBracket
from jacopy.library.theorem_book import theorem_book


## Setup

Bivector $\pi$ and three generic 1-forms $\alpha,\beta,\gamma$ (each declared with shifted SN degree 1).

In [2]:
reg = PropertyRegistry()

pi    = Symbol('π'); reg.declare(pi,    Graded(degree=1))
alpha = Symbol('α'); reg.declare(alpha, Graded(degree=1))
beta  = Symbol('β'); reg.declare(beta,  Graded(degree=1))
gamma = Symbol('γ'); reg.declare(gamma, Graded(degree=1))

P = PoissonBracket.from_bivector(pi)
K = P.koszul_derived
K


DerivedBracket('{·,·}_π,♯')

## LHS, cyclic Koszul Jacobi obstruction

In [3]:
lhs = K.graded_jacobi_obstruction(alpha, beta, gamma, registry=reg)
for i, c in enumerate(lhs.children):
    print(f'[{i}] {c}')


[0] (-{·,·}_π,♯(α, {·,·}_π,♯(β, γ)))
[1] (-{·,·}_π,♯(β, {·,·}_π,♯(γ, α)))
[2] (-{·,·}_π,♯(γ, {·,·}_π,♯(α, β)))


## Closure, Derived Bracket Theorem in one step

`prove_koszul_jacobi_reduction` returns a `ProofChain` whose single top-level step rewrites the cyclic sum to the SN handle $[\pi,\pi]_{SN}$, the same handle that 2g lands on. No residue.

In [4]:
chain = P.prove_koszul_jacobi_reduction(alpha, beta, gamma, registry=reg)
print(f'steps  : {len(chain)}')
print(f'initial: {chain.initial}')
print(f'final  : {chain.final}')


steps  : 1
initial: ((-{·,·}_π,♯(α, {·,·}_π,♯(β, γ))) + (-{·,·}_π,♯(β, {·,·}_π,♯(γ, α))) + (-{·,·}_π,♯(γ, {·,·}_π,♯(α, β))))
final  : [·,·]_SN(π, π)


In [5]:
step, = chain.steps
print(f'rule         : {step.rule}')
print(f'justification: {step.justification}')
print(f'before       : {step.before}')
print(f'after        : {step.after}')


rule         : DerivedBracketTheorem
justification: Jacobi on {·,·}_π,♯ ⟺ [π, π]_SN = 0 (Derived Bracket Theorem)
before       : ((-{·,·}_π,♯(α, {·,·}_π,♯(β, γ))) + (-{·,·}_π,♯(β, {·,·}_π,♯(γ, α))) + (-{·,·}_π,♯(γ, {·,·}_π,♯(α, β))))
after        : [·,·]_SN(π, π)


Closure check, chain ends exactly at $[\pi,\pi]_{SN}$:

In [6]:
expected_handle = BracketApply(default_sn, pi, pi)
print('final           :', chain.final)
print('expected handle :', expected_handle)
print('match           :', chain.final == expected_handle)
print('residue         :', None)


final           : [·,·]_SN(π, π)
expected handle : [·,·]_SN(π, π)
match           : True
residue         : None


## Both sides land at the same handle

Function side (2g) and form side (2f) hit the *same* `BracketApply([·,·]_SN, π, π)`, the universal Poisson obstruction is view-independent:

In [7]:
freg = PropertyRegistry()
f = Symbol('f'); freg.declare(f, Graded(degree=-1))
g = Symbol('g'); freg.declare(g, Graded(degree=-1))
h = Symbol('h'); freg.declare(h, Graded(degree=-1))
func_chain = P.prove_jacobi_reduction(f, g, h, registry=freg)
form_chain = chain
print('function-side handle :', func_chain.final)
print('form-side handle     :', form_chain.final)
print('same handle          :', func_chain.final == form_chain.final)


function-side handle : [·,·]_SN(π, π)
form-side handle     : [·,·]_SN(π, π)
same handle          : True


## Theorem-book record

The form-side reduction is seeded as `poisson_koszul_jacobi`.

In [8]:
thm = theorem_book.get('poisson_koszul_jacobi')
print('name      :', thm.name)
print('statement :', thm.statement)
print('axioms    :')
for ax in thm.from_axioms:
    print('   -', ax)


name      : poisson_koszul_jacobi
statement : Koszul Jacobi on {·,·}_π cyclic sum = 0 when [π, π]_SN = 0
axioms    :
   - Derived Bracket Theorem
   - π^♯ = Sharp(π) as form-lift anchor
   - [π, π]_SN = 0 (Poisson hypothesis)


## Conclusion

$$
\boxed{\;\sum_{\text{cyc}}[\alpha,[\beta,\gamma]_K]_K \;\stackrel{\text{Derived Bracket Thm}}{=}\; \tfrac12\,[\pi,\pi]_{SN}(\alpha,\beta,\gamma).\;}
$$

One step, no residue. The form side (2f) and the function side (2g) meet at the same universal handle $[\pi,\pi]_{SN}$, Jacobi closes on both views simultaneously, exactly when this handle vanishes.